# rearrange-as-sequential-layer — ex1: build a Conv-flatten-Linear pipeline with Rearrange layer (no forward boilerplate)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `rearrange-as-sequential-layer`. Running the final beacon cell reports progress against the `Einops: Rearrange as nn.Sequential layer` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
from einops.layers.torch import Rearrange

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Rearrange as nn.Sequential layer` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`rearrange-as-sequential-layer`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "rearrange-as-sequential-layer"
DD_SUBTOPIC = "Einops: Rearrange as nn.Sequential layer"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `einops.layers.torch.Rearrange` as nn.Sequential layer — quick refresher

Inside a `forward()` you'd write `einops.rearrange(x, 'b c h w -> b (c h w)')`. But when composing inside `nn.Sequential`, you need a Module, not a function. `einops.layers.torch.Rearrange` is the answer:

```python
from einops.layers.torch import Rearrange

model = nn.Sequential(
    nn.Conv2d(3, 32, 3, padding=1),
    nn.ReLU(),
    Rearrange('b c h w -> b (c h w)'),    # the 'Flatten' step
    nn.Linear(32 * 28 * 28, 10),
)
```

**Why this beats `nn.Flatten()`.** Flatten is opaque — you have to read the docs to remember its `start_dim` / `end_dim` semantics. `Rearrange` puts the shape transformation in the source as an algebraic string; the next reader sees `b c h w -> b (c h w)` and knows exactly what shape comes out.

**There are three sibling Modules.** `Rearrange`, `Reduce`, and the rare `EinMix`. All live in `einops.layers.torch` (or `.tensorflow`, `.flax`). Use `Reduce` for global-average-pool / channel-mean / softmax-stabilize inside Sequential, same way.

**Gotcha — not the same import path.** `from einops import rearrange` gets you the *function*. `from einops.layers.torch import Rearrange` (capital R, different module) gets you the *layer*. They share the same string grammar; only the wrapping differs.

### Exercise 1 — build a Conv-flatten-Linear pipeline with Rearrange layer (no forward boilerplate)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Wrap einops.layers.torch.Rearrange inside an nn.Sequential so a Conv → ReLU → flatten → Linear pipeline composes without writing a custom forward.
> Keywords: einops, Rearrange, layer, Sequential, Flatten
> ```

**KCs targeted:** `rearrange-layer-import-path`, `rearrange-in-sequential-composes`

Implement `ex1_conv_classifier(in_channels, height, width, num_classes)` — a minimal image classifier that uses `einops.layers.torch.Rearrange` as a layer inside `nn.Sequential`, eliminating the need for a custom `forward()`:

1. Import `from einops.layers.torch import Rearrange` (capital R — this is the Module form, not the `einops.rearrange` function).
2. Build:
   ```
   nn.Sequential(
       nn.Conv2d(in_channels, 8, kernel_size=3, padding=1),  # (B, 8, H, W)
       nn.ReLU(),
       Rearrange('b c h w -> b (c h w)'),                     # (B, 8*H*W)
       nn.Linear(8 * height * width, num_classes),            # (B, num_classes)
   )
   ```
3. Return the `nn.Sequential` instance directly — DO NOT wrap it in your own Module subclass. The point of this drill is that `Rearrange` makes the wrapper unnecessary.

Input shape: `(B, in_channels, height, width)`. Output shape: `(B, num_classes)`.

**Why this beats `nn.Flatten()`.** Flatten's signature requires you to remember `start_dim=1, end_dim=-1` to keep the batch axis. `Rearrange('b c h w -> b (c h w)')` puts the shape transformation in the source — anyone reading the code knows exactly what shape comes out without consulting docs.

**Critical import path.** `einops.rearrange` is the FUNCTION (use inside `forward`). `einops.layers.torch.Rearrange` is the MODULE (use inside `nn.Sequential`). They share the same string grammar; only the wrapping differs.

In [ ]:
def ex1_conv_classifier(in_channels: int, height: int, width: int, num_classes: int):
    """Return an nn.Sequential pipeline using einops.layers.torch.Rearrange as the flatten layer."""
    raise NotImplementedError()


def _test_ex1():
    from einops.layers.torch import Rearrange

    model = ex1_conv_classifier(in_channels=3, height=8, width=8, num_classes=10)

    # The model must be an nn.Sequential — NOT a custom Module wrapping one.
    assert isinstance(model, t.nn.Sequential), (
        f'expected nn.Sequential, got {type(model).__name__}. '
        'The whole point is that Rearrange-as-layer means you do not need a custom Module.'
    )

    # Sequential should contain exactly 4 layers in the right order.
    assert len(model) == 4, f'expected 4 layers (Conv, ReLU, Rearrange, Linear), got {len(model)}'
    assert isinstance(model[0], t.nn.Conv2d), f'layer 0 should be Conv2d, got {type(model[0]).__name__}'
    assert isinstance(model[1], t.nn.ReLU), f'layer 1 should be ReLU, got {type(model[1]).__name__}'
    assert isinstance(model[2], Rearrange), (
        f'layer 2 should be einops.layers.torch.Rearrange, got {type(model[2]).__name__}. '
        'Did you use nn.Flatten() instead? This drill specifically tests the Rearrange-as-layer pattern.'
    )
    assert isinstance(model[3], t.nn.Linear), f'layer 3 should be Linear, got {type(model[3]).__name__}'

    # Layer dimensions.
    assert model[0].in_channels == 3
    assert model[0].out_channels == 8
    assert model[3].in_features == 8 * 8 * 8  # 8 channels * 8 H * 8 W
    assert model[3].out_features == 10

    # Forward — shape check.
    x = t.randn(4, 3, 8, 8, generator=t.Generator().manual_seed(0))
    y = model(x)
    assert y.shape == (4, 10), f'expected (4, 10), got {tuple(y.shape)}'

    # Verify the Rearrange step DID flatten the conv output (not just transpose).
    # Run partial pipeline up through the Rearrange and check the shape.
    intermediate = model[0:3](x)  # Conv → ReLU → Rearrange
    assert intermediate.shape == (4, 8 * 8 * 8), (
        f'after Rearrange the shape should be (B, c*h*w) = (4, 512), got {tuple(intermediate.shape)}. '
        'Check your Rearrange string — should be "b c h w -> b (c h w)".'
    )

    # Test with different spatial dims to confirm the Rearrange string generalizes.
    model2 = ex1_conv_classifier(in_channels=1, height=4, width=12, num_classes=3)
    x2 = t.randn(2, 1, 4, 12, generator=t.Generator().manual_seed(1))
    y2 = model2(x2)
    assert y2.shape == (2, 3), f'second config wrong: {tuple(y2.shape)}'

    # Confirm there's no custom Module wrapping — parameters should belong to the
    # top-level Sequential, not a nested 'model.net.*' path.
    named = dict(model.named_parameters())
    # Expected: '0.weight', '0.bias' (Conv2d), '3.weight', '3.bias' (Linear). ReLU + Rearrange have no params.
    expected_param_names = {'0.weight', '0.bias', '3.weight', '3.bias'}
    assert set(named.keys()) == expected_param_names, (
        f'expected param names {expected_param_names}, got {set(named.keys())}. '
        'If you see net.0.weight etc., you wrapped Sequential inside a custom Module — undo that.'
    )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_conv_classifier(in_channels: int, height: int, width: int, num_classes: int):
    from einops.layers.torch import Rearrange
    return t.nn.Sequential(
        t.nn.Conv2d(in_channels, 8, kernel_size=3, padding=1),
        t.nn.ReLU(),
        Rearrange('b c h w -> b (c h w)'),
        t.nn.Linear(8 * height * width, num_classes),
    )
```

**Why `Rearrange` is a Module.** `einops.layers.torch.Rearrange.__init__` stores the einops pattern string; its `forward` calls `einops.rearrange(x, self.pattern)`. Because it subclasses `nn.Module`, `nn.Sequential` accepts it and auto-pipes the output. Zero parameters, no `__init__` work needed by you.

**The full sibling set in `einops.layers.torch`.** `Rearrange` (the layer form of `einops.rearrange`), `Reduce` (the layer form of `einops.reduce` — handy for global average pool inside Sequential), and `EinMix` (a learnable Einsum-based linear layer). All three accept the same string grammar as their function-form counterparts.

**ARENA's `make_cnn` uses this exact pattern** — Conv → BN → ReLU stages followed by `Rearrange('b c h w -> b (c h w)')` and a final Linear. Knowing the Rearrange-as-layer trick is what lets the whole thing fit in a single `nn.Sequential` with no custom Module wrapper.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()